### Roll-numbers


### Instructions
 * Fill in the roll-number in the cell above.
 * Code must be submitted in Python in jupyter notebooks. We highly recommend using anaconda/miniconda distribution or at the minimum, virtual environments for this assignment.
 * All the code and result files should be uploaded in the github classroom.
 * For this assignment, you will be using Open3D  extensively. Refer to [Open3D](http://www.open3d.org/docs/release/) documentation.
 *  Most of the questions require you to **code your own functions** unless there is a need to call in the abilities of the mentioned libraries, such as Visualisation from Open3D. Make sure your code is modular since you will be reusing them for future assignments. All the functions related to transformation matrices, quaternions, and 3D projection are expected to be coded by you.
 *  All the representations are expected to be in a right-hand coordinate system.
<!--  * Answer to the descriptive questions should be answered in your own words. Copy-paste answers will lead to penalty. -->
 * You could split the Jupyter Notebook cells where TODO is written, but please try to avoid splitting/changing the structure of other cells.
 * All the visualization should be done inside the notebook unless specified otherwise.
 * Plagiarism will lead to heavy penalty.
 * Commit the notebooks in the repo and any other results files under the result folder in the GitHub Classroom repo. 
 * Commits past the deadline will not be considered.
 * This is a group assignment. Discussions are encouraged but any sharing of code among different teams will be penalized.

### Note
If it isn't obvious enough already, we have access to the same "*resources*" as you, and any plagiarism of code will be heavily penalised, no exceptions. We suggest you do the assignments honestly as it'll improve your own understanding.


# Q1: Transformations and Projections on Autonomous Driving Dataset (20 Points)

In this question, you will work with real world autonomous driving dataset (sequence in Waymo dataset). The dataset has LiDAR point clouds, images. You are required to demonstrate: 

**I. Various transformations of rotation matrices as described in below tasks.**

**II. Visualization as a result of above transformations in Open3D**

## Given data:

1.) `LiDAR Point Clouds` : Stored at each timestep in the folder `lidar`. The point clouds are provided in the ego frame attached to lidar sensor (vehicle's reference frame).

2.) `Images` : Stored at each timestep in the folder `images`. 

**Naming Convention** : {timestep}_{cam_no}.jpg where timestep is specified in 3 digits and cam_no : [0, 1, 2] indicates centre, left and right camera respectively.

3.) `Camera-to-Ego Transformations`: Stored in the folder `cam2ego`, which converts points from each camera's reference frame to the vehicle's (or ego) reference frame.

4.) `Ego-to-World Transformations`: Stored in the folder `ego2world`, which converts points from the vehicle's reference frame to the world frame W.

5.) `Camera Intrinsics`: Stored in the folder `intrinsics` provided for 3 cameras.




### Helper functions to read lidar data and camera instrinsics are provided below

In [3]:
# Helper function to read instrinsic matrix

import numpy as np
import os 

def read_intrinsic(timestep):
    intrinsic = np.loadtxt(f"sample_intrinsic_{timestep}.txt")
    fx, fy, cx, cy = intrinsic[0], intrinsic[1], intrinsic[2], intrinsic[3]
    intrinsic_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

read_intrinsic(1)

ModuleNotFoundError: No module named 'numpy'

In [3]:
# Helper function to read lidar data at timestep 0 (same logic to read lidars at all remaining timesteps)

lidar_data = np.memmap('sample_lidar_data_000.bin',
                dtype=np.float32,
                mode="r",
            ).reshape(-1, 14)   # (165454, 14)

lidar_origins = lidar_data[:, :3]
lidar_points = lidar_data[:, 3:6]   # (165454, 3)
lidar_ids = lidar_data[:, -1]   # (165454,)

# Lidar points to be used 
print(lidar_points.shape)

(165454, 3)



**Note:** Even though Waymo dataset has 5 cameras, you are given the dataset corresponding to middle 3 cameras only. Please ignore other 2 cameras.

## Notation for tasks:

a.) `Global Reference Frame G`: Defined as the first ego frame (i.e., the translation vector of ego2world[0] is the origin of frame G in world frame W).

World Frame W: A fixed world reference frame.

b.) `Ego Frame`: Attached to the LiDAR and changes as the vehicle moves.

c.) `Camera Frames`: Each of the 5 cameras has its own frame, which changes as the vehicle moves.

Note: Axis directions of `Ego Frame` and `Camera Frames` are aligned with the Waymo Coordinate System (LiDAR) described below

## Coordinate Systems:

**OpenCV Coordinate System:** x right, y down, z front.

**Waymo Coordinate System (LiDAR):** x front, y left, z up.


![Waymo Setup](./waymo_setup.jpg "Waymo Setup")

Link to dataset (one sequence) : https://drive.google.com/drive/folders/17YDx2Yn1KmPjmlaHsoFz4Jpa8zpgovO2?usp=drive_link

If you want to try on other sequences as well, please refer to : https://waymo.com/open/

### `Task 1`. Transformations of LiDAR Point Clouds (10 points)

**Instructions:** 

Transform the LiDAR point clouds at all timesteps to the global reference frame G. Concatenate these transformed point clouds.
    
Visualization: Use Open3D to visualize the concatenated point cloud in the global reference frame G. Also, display the concatenation process at every timestep starting from first point cloud





In [ ]:
##############################################################################
# TODO: TASK 1
##############################################################################\

import numpy as np
import open3d as o3d
from glob import glob


def load_lidar_bin(file):
    """
    Load lidar bin file of shape (N,14).
    Return xyz points (N,3).
    """
    lidar_data = np.memmap(file, dtype=np.float32, mode="r").reshape(-1, 14)
    lidar_points = lidar_data[:, 3:6]  # only xyz
    return lidar_points


def load_ego2world(file):
    """
    Load ego2world matrix from txt file.
    Returns 4x4 transformation matrix.
    """
    mat = np.loadtxt(file).reshape(4, 4)
    return mat


def transform_points(points, T):
    """
    Transform points (N,3) using homogeneous 4x4 transform T.
    """
    N = points.shape[0]
    homo_points = np.hstack([points, np.ones((N, 1))])  # (N,4)
    transformed = (T @ homo_points.T).T[:, :3]
    return transformed


lidar_files = sorted(glob("../080_MR/lidar/*.bin"))
ego2world_files = sorted(glob("../080_MR/ego2world/*.txt"))

assert len(lidar_files) == len(ego2world_files), "Mismatch between LiDAR and ego2world files"


all_points = []
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="LiDAR Concatenation", width=800, height=600)
pcd = o3d.geometry.PointCloud()

for i, (lf, tf) in enumerate(zip(lidar_files, ego2world_files)):
    lidar_pts = load_lidar_bin(lf)
    T = load_ego2world(tf)
    lidar_global = transform_points(lidar_pts, T)

    all_points.append(lidar_global)

    # Update visualization step by step
    concat_points = np.vstack(all_points)
    pcd.points = o3d.utility.Vector3dVector(concat_points)

    if i == 0:
        vis.add_geometry(pcd)
    else:
        vis.update_geometry(pcd)

    vis.poll_events()
    vis.update_renderer()
    
    
concat_points = np.vstack(all_points)
np.save("concat_points.npy", concat_points)
print(f"[INFO] Saved concatenated global LiDAR points -> concat_points.npy")


vis.run()
vis.destroy_window()


In [6]:
import numpy as np, open3d as o3d

pts = np.load("concat_points.npy")
print("Shape:", pts.shape, "min:", pts.min(axis=0), "max:", pts.max(axis=0))

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
o3d.visualization.draw_geometries([pcd])


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Shape: (32894768, 3) min: [-1548.19091295 14145.49978211   -16.89023309] max: [-1.30910818e+03  1.43197516e+04 -2.74772723e+00]


### `Task 2`. Projecting LiDAR Point Clouds onto images (10 points)

**Instructions:**

Transform the concatenated point cloud from task 1 to the frame of each of the 3 cameras at timesteps `0, 30, and 42`. 
    
Project these transformed point clouds onto the respective camera frames using the provided camera intrinsics. Concatenated point cloud would be very dense, so randomly select arbitrary number of points for better visualization. 

**Projected image pixel x : K * X_3d where X_3d is the 3d point in camera frame.**
    
Visualization: Overlay the projected points onto the camera images and visualize them.

**For example:** Overlayed concatenated point cloud on camera `000_0.png` and `030_2.png` are shown below

<table><tr>
<td> <img src="./projected_000_0.png" alt="Drawing" style="width: 750px;"/> </td>
<td> <img src="./projected_030_2.png" alt="Drawing" style="width: 750px;"/> </td>
</tr></table>



In [ ]:
##############################################################################
# TODO: TASK 2 (fixed)
##############################################################################
# Project concatenated world-frame point cloud onto camera images


import numpy as np
import cv2
import os

def load_matrix(file):
    """Load 4x4 matrix from txt (extrinsics)."""
    return np.loadtxt(file)

def load_intrinsics(file):
    """Load 3x3 intrinsics matrix from txt."""
    
    vals = np.loadtxt(file)
    fx, fy, cx, cy = vals[:4]
    K = np.array([[fx, 0, cx],
                  [0, fy, cy],
                  [0,  0,  1]])
    # distortion = vals[4:]   # (k1,k2,p1,p2,k3) if you want to undistort later
    return K

def transform_points(points, T):
    """Apply 4x4 transform to Nx3 points."""
    homog = np.hstack((points, np.ones((points.shape[0], 1))))
    return (T @ homog.T).T[:, :3]

def waymo_to_opencv(points):
    """Convert Waymo coord (x front, y left, z up) → OpenCV (x right, y down, z front)."""
    R = np.array([[0,0,1],
                  [1,0,0],
                  [0,-1,0]])
    return (R @ points.T).T

def project_points(points, K, img_shape):
    """Project 3D points in OpenCV cam coords to 2D image plane + return depths."""
    X, Y, Z = points[:,0], points[:,1], points[:,2]
    mask = Z > 0  # keep only in front of camera
    X, Y, Z = X[mask], Y[mask], Z[mask]
    
    homog = np.vstack((X, Y, Z))
    pixels = (K @ homog).T
    u = pixels[:,0] / pixels[:,2]
    v = pixels[:,1] / pixels[:,2]

    # filter inside image
    h, w = img_shape[:2]
    valid = (u >= 0) & (u < w) & (v >= 0) & (v < h)

    u, v, Z = u[valid], v[valid], Z[valid]

    uv = np.vstack((u,v)).T
    depths = Z  # keep depth for coloring

    return uv, depths


def visualize_projection(points_global, ego2world_t, cam2ego, K, img, save_path, num_samples=50000):
    # global → ego(t)
    world2ego = np.linalg.inv(ego2world_t)
    points_ego = transform_points(points_global, world2ego)

    # ego → cam
    ego2cam = np.linalg.inv(cam2ego)
    points_cam = transform_points(points_ego, ego2cam)

    # Waymo → OpenCV
    points_cv = waymo_to_opencv(points_cam)

    # Project (get uv and depths)
    uv, depths = project_points(points_cv, K, img.shape)

    # Sample subset
    if len(uv) > num_samples:
        idx = np.random.choice(len(uv), num_samples, replace=False)
        uv, depths = uv[idx], depths[idx]

    # Normalize depths to [0,255] for coloring
    depths = np.clip(depths, 0, 80)  # clamp to 80m
    colors = (255 * (1 - depths/80)).astype(np.uint8)

    # Overlay
    for (u,v), c in zip(uv.astype(int), colors):
        cv2.circle(img, (u,v), 2, (int(c), 0, 255-int(c)), -1)

    cv2.imwrite(save_path, img)
    print(f"Saved {save_path} with {len(uv)} points")


# -------------------- MAIN --------------------

# Load concatenated lidar point cloud (already in Global frame)
points_global = np.load("concat_points.npy")

# Timesteps and cameras to visualize
timesteps = [0, 30, 42]
cameras = [0, 1, 2]

base_dir = "../080_MR/"

for t in timesteps:
    ego2world_t = load_matrix(f"{base_dir}ego2world/{t:03d}.txt")  # still needed for global→ego

    for cam in cameras:
        cam2ego = load_matrix(f"{base_dir}cam2ego/{cam}.txt")
        K = load_intrinsics(f"{base_dir}intrinsics/{cam}.txt")
        img_path = os.path.join(base_dir, "images", f"{t:03d}_{cam}.jpg")
        img = cv2.imread(img_path)

        save_path = f"results/proj_cam{cam}_t{t}.png"
        visualize_projection(points_global, ego2world_t, cam2ego, K, img, save_path)


Saved results/proj_cam0_t0.png with 50000 points
Saved results/proj_cam1_t0.png with 50000 points
Saved results/proj_cam2_t0.png with 50000 points
Saved results/proj_cam0_t30.png with 50000 points
Saved results/proj_cam1_t30.png with 50000 points
Saved results/proj_cam2_t30.png with 50000 points
Saved results/proj_cam0_t42.png with 50000 points
Saved results/proj_cam1_t42.png with 50000 points
Saved results/proj_cam2_t42.png with 50000 points


## Bonus

### `Task 3`. Compute Depth Image from Projected Point Cloud in camera frame (5 points)

**Instructions:**

Using the projected point clouds to camera frame from task 2, visualize the depth image by considering only the z-coordinate of the projected points in the camera frame.

Visualization: Display the depth image for each of the 3 cameras at timesteps `0, 30, and 42` alongside the corresponding RGB image.

In [17]:
##############################################################################
# TODO: TASK 3
##############################################################################

#### Note: You might be asked to show the above results for different timesteps and from one of the 3 cameras during evaluation/viva.

# Q2: Various Representations for Rotations and Gimbal lock (15 points)


#### 2.1 Euler angles (2.5 points)

a. Write a function that returns a rotation matrix given the angles (𝛼, 𝛽, 𝛾) = (3π/5, π/9, 5π/6) in radians (X-Y-Z). Do not use inbuilt functions.

In [11]:
#! general functionality class
import numpy as np

class rotation:
    def __init__(self):
        self.quat = np.array([1, 0, 0, 0])  # W, X, Y, Z
        self.mat = np.eye(3)
        self.euler = np.array([0, 0, 0])      # ROLL, PITCH, YAW
        self.rotvec = np.array([0, 0, 0])     # ROTVEC

    def as_matrix(self):
        self.to_matrix()
        return self.mat

    def as_quat(self, MAT):
        """Converts rotation matrix to quaternion"""
        self.from_matrix(MAT)
        return self.quat

    def as_euler(self):
        self.to_euler()
        return self.euler

    def as_rotvec(self):
        self.to_rotvec()
        return self.rotvec

    def to_matrix(self):
        """Convert quaternion to matrix"""
        W, X, Y, Z = self.quat
        self.mat = np.array([[1 - 2*Y**2 - 2*Z**2, 2*X*Y - 2*Z*W, 2*X*Z + 2*Y*W],
                             [2*X*Y + 2*Z*W, 1 - 2*X**2 - 2*Z**2, 2*Y*Z - 2*X*W],
                             [2*X*Z - 2*Y*W, 2*Y*Z + 2*X*W, 1 - 2*X**2 - 2*Y**2]])

    def to_euler(self):
        """Convert quaternion to Euler angles"""
        W, X, Y, Z = self.quat
        self.euler = np.array([
            np.arctan2(2*(W*X + Y*Z), 1 - 2*(X**2 + Y**2)),
            np.arcsin(2*(W*Y - Z*X)),
            np.arctan2(2*(W*Z + X*Y), 1 - 2*(Y**2 + Z**2))
        ])

    def to_rotvec(self):
        """Convert quaternion to rotation vector"""
        W, X, Y, Z = self.quat
        THETA = 2 * np.arccos(W)
        if THETA < 1e-6:
            self.rotvec = np.array([0, 0, 0])
        else:
            self.rotvec = THETA * np.array([X, Y, Z]) / np.sin(THETA / 2)
        return self.rotvec
        
    def from_matrix(self, MAT):
        """Convert matrix to quaternion"""
        self.quat = np.zeros(4)
        T = np.trace(MAT)
        if T > 0:
            S = np.sqrt(T + 1.0) * 2
            self.quat[0] = 0.25 * S
            self.quat[1] = (MAT[2, 1] - MAT[1, 2]) / S
            self.quat[2] = (MAT[0, 2] - MAT[2, 0]) / S
            self.quat[3] = (MAT[1, 0] - MAT[0, 1]) / S
        else:
            if MAT[0, 0] > MAT[1, 1] and MAT[0, 0] > MAT[2, 2]:
                S = np.sqrt(1.0 + MAT[0, 0] - MAT[1, 1] - MAT[2, 2]) * 2
                self.quat[0] = (MAT[2, 1] - MAT[1, 2]) / S
                self.quat[1] = 0.25 * S
                self.quat[2] = (MAT[0, 1] + MAT[1, 0]) / S
                self.quat[3] = (MAT[0, 2] + MAT[2, 0]) / S
            elif MAT[1, 1] > MAT[2, 2]:
                S = np.sqrt(1.0 + MAT[1, 1] - MAT[0, 0] - MAT[2, 2]) * 2
                self.quat[0] = (MAT[0, 2] - MAT[2, 0]) / S
                self.quat[1] = (MAT[0, 1] + MAT[1, 0]) / S
                self.quat[2] = 0.25 * S
                self.quat[3] = (MAT[1, 2] + MAT[2, 1]) / S
            else:
                S = np.sqrt(1.0 + MAT[2, 2] - MAT[0, 0] - MAT[1, 1]) * 2
                self.quat[0] = (MAT[1, 0] - MAT[0, 1]) / S
                self.quat[1] = (MAT[0, 2] + MAT[2, 0]) / S
                self.quat[2] = (MAT[1, 2] + MAT[2, 1]) / S
                self.quat[3] = 0.25 * S
                
    def from_quat(self, QUAT):
        self.quat = QUAT / np.linalg.norm(QUAT)
        
    def from_euler(self, EULER):
        """Convert Euler angles to quaternion"""
        ROLL, PITCH, YAW = EULER
        CY = np.cos(YAW * 0.5)
        SY = np.sin(YAW * 0.5)
        CP = np.cos(PITCH * 0.5)
        SP = np.sin(PITCH * 0.5)
        CR = np.cos(ROLL * 0.5)
        SR = np.sin(ROLL * 0.5)

        self.quat = np.array([
            CR * CP * CY + SR * SP * SY,
            SR * CP * CY - CR * SP * SY,
            CR * SP * CY + SR * CP * SY,
            CR * CP * SY - SR * SP * CY
        ])

    def from_rotvec(self, ROTVEC):
        """Convert rotation vector to quaternion"""
        THETA = np.linalg.norm(ROTVEC)
        if THETA < 1e-6:
            self.quat = np.array([1, 0, 0, 0])
        else:
            AXIS = ROTVEC / THETA
            self.quat = np.array([
                np.cos(THETA / 2),
                np.sin(THETA / 2) * AXIS[0],
                np.sin(THETA / 2) * AXIS[1],
                np.sin(THETA / 2) * AXIS[2]
            ])
            
    ############################# DIRECT CONVERSIONS AS PER QUESTION REQUIREMENTS ##################################
    
    def euler_to_mat(self, ANGLES, order="xyz"):
        X, Y, Z = ANGLES
        CX, CY, CZ = np.cos([X, Y, Z])
        SX, SY, SZ = np.sin([X, Y, Z])

        RX = np.array([[1, 0, 0],
                       [0, CX, -SX],
                       [0, SX, CX]])

        RY = np.array([[CY, 0, SY],
                       [0, 1, 0],
                       [-SY, 0, CY]])

        RZ = np.array([[CZ, -SZ, 0],
                       [SZ, CZ, 0],
                       [0, 0, 1]])

        if order == "xyz":
            self.mat = RZ @ RY @ RX   # intrinsic XYZ
        elif order == "zyx":
            self.mat = RX @ RY @ RZ   # intrinsic ZYX
        else:
            raise ValueError("Order not implemented")

        return self.mat
    
    def rotvec_to_matrix(self, ROTVEC):
        THETA = np.linalg.norm(ROTVEC)
        if THETA < 1e-6:
            return np.eye(3)
        K_VEC = ROTVEC / THETA
        K_MAT = np.array([[0, -K_VEC[2], K_VEC[1]],
                          [K_VEC[2], 0, -K_VEC[0]],
                          [-K_VEC[1], K_VEC[0], 0]])
        self.mat = np.eye(3) + np.sin(THETA) * K_MAT + (1 - np.cos(THETA)) * (K_MAT @ K_MAT)
        return self.mat
    
    def quat_to_matrix(self, QUAT):
        W, X, Y, Z = QUAT
        self.mat = np.array([[1 - 2 * (Y**2 + Z**2), 2 * (X * Y - W * Z), 2 * (X * Z + W * Y)],
                             [2 * (X * Y + W * Z), 1 - 2 * (X**2 + Z**2), 2 * (Y * Z - W * X)],
                             [2 * (X * Z - W * Y), 2 * (Y * Z + W * X), 1 - 2 * (X**2 + Y**2)]])
        return self.mat

In [8]:
##############################################################################
# TODO: Do tasks described in 2.1 (a)
##############################################################################

#### 2.2 Equivalent angle–axis representation (2.5 points) 

 Write a function to convert equivalent angle–axis representation (with a general axis and angle) to matrix form and vice versa. \
Try it for $\theta = 5\pi/6$ and axis $K= [1, 2, 3]^T $

In [ ]:
##############################################################################
# TODO: Do tasks described in 2.2 
##############################################################################

#### 2.3 Gimbal lock (5 points)

Show an example where a Gimbal lock occurs and visualize the Gimbal lock on the given point cloud, data/toothless.ply. You have to show the above by animation (rotation along each axis one by one).

**Hint:** 
Create 3 disks perpendicular to each other representing axes for local frame of object. Show that in certain configuration, due to use of Euler angles we can lose a degree of freedom. 

Use Open3D's non-blocking visualization and discretize the rotation to simulate the animation. For example, if you want to rotate by 20° around a particular axis, do so in increments of 5° 4 times to make it look like an animation.

In [ ]:
##############################################################################
# TODO: Do tasks described in 2.3
##############################################################################

#### 2.4: Quaternions (5 points)

a. Convert a rotation matrix to quaternion and vice versa. Do not use inbuilt libraries for this question.

b. Perform matrix multiplication of two 3×3 rotation matrices and perform the same transformation in the quaternion space. Verify if the final transformation obtained in both cases is the same.

c. Try to interpolate any given model between two rotation matrices and visualize!

In [2]:
##############################################################################
# TODO: Do tasks described in 2.4 (a)
##############################################################################

In [3]:
##############################################################################
# TODO: Do tasks described in 2.4 (b)
##############################################################################

In [8]:
##############################################################################
# TODO: Do tasks described in 2.4 (c)
##############################################################################

import open3d as o3d
import numpy as np
import copy
import time
from scipy.spatial.transform import Rotation as R, Slerp

class animate_tf:
    def __init__(self,pcd,func=None):
        self.pcd = pcd
        self.points = np.asarray(pcd.points)
        
        self.state = {
            "alpha": 0.0,
            "dir": 1,
            "step": 0.01
        }
        
        self.interpol_func = func

    def tf_pcd(self, points, tf):
        """Transform Nx3 numpy points by homogeneous tf, return Nx3 numpy array."""
        ones = np.ones((points.shape[0], 1))
        points_hom = np.hstack((points, ones))
        transformed = (points_hom @ tf.T)[:, :3]
        return transformed

    def interpolate_tf(self, *args):
        if len(args) == 3:
            T1, T2, alpha = args
            return (1.0 - alpha) * T1 + alpha * T2
        elif len(args) == 1 and callable(args[0]):
            func = args[0]
            return func(self.T1, self.T2, self.state["alpha"])
        else:
            raise ValueError("Invalid arguments. Provide either (T1, T2, alpha) or a single callable function.")
        
    def setup(self,T1,T2):
        self.T1 = T1
        self.T2 = T2
        
    def animate(self):
        vis = o3d.visualization.Visualizer()
        vis.create_window()

        pcd1 = o3d.geometry.PointCloud()
        pcd1.points = o3d.utility.Vector3dVector(self.tf_pcd(self.points, self.T1))
        vis.add_geometry(pcd1)

        pcd2 = o3d.geometry.PointCloud()
        pcd2.points = o3d.utility.Vector3dVector(self.tf_pcd(self.points, self.T2))
        vis.add_geometry(pcd2)

        pcd_interp = o3d.geometry.PointCloud()
        pcd_interp.points = o3d.utility.Vector3dVector(self.tf_pcd(self.points, self.T1))  # start at T1
        vis.add_geometry(pcd_interp)

        # camera (optional)
        ctr = vis.get_view_control()
        ctr.set_lookat([0, 0, 0])
        ctr.set_front([1, 1, 1])
        ctr.set_up([0, 0, 1])
        ctr.set_zoom(1.0)

        def animation_callback(vis):
            a = self.state["alpha"] + self.state["dir"] * self.state["step"]
            if a >= 1.0:
                a = 1.0
                self.state["dir"] = -1
            elif a <= 0.0:
                a = 0.0
                self.state["dir"] = 1
            self.state["alpha"] = a

            T = self.interpolate_tf(T1, T2, self.state["alpha"]) if self.interpol_func is None else self.interpolate_tf(self.interpol_func)
            new_pts = self.tf_pcd(self.points, T)
            pcd_interp.points = o3d.utility.Vector3dVector(new_pts)

            vis.update_geometry(pcd_interp)
            return False

        vis.register_animation_callback(animation_callback)

        vis.run()
        vis.destroy_window()


T1 = np.array([[1, 0, 0, -1000], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
T2 = np.array([[1, 0, 0, 1000], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])

points = o3d.io.read_point_cloud("toothless.ply")
anim = animate_tf(points)
anim.setup(T1, T2)
anim.animate()

# Q3: Interpolation between transformations (20 points)

## 3.1 - Different ways to interpolate
Given 2 random transformation matrices, interpolate the given point cloud **toothless.ply** from `T2` to `T1` in 3 different methods and visualize it.

We will use the `generateTransformation()` function to generate a random Transformation matrix. You can write your own `generateTransformation()` function for testing, but we will replace it with our own so make sure that your code works for general cases.

Ensure that your visualization shows the starting and ending configurations during interpolation.

Your final output should look something like this:
![Visualization](./out.gif)

In [4]:
import numpy as np

def generateTransformation():
    angle = np.random.uniform(0, 2*np.pi)
    axis = np.random.randn(3)
    axis /= np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    Rm = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    t = np.random.uniform(-1000, 1000, (3,))
    T = np.eye(4)
    T[:3, :3] = Rm
    T[:3, 3] = t
    return T    

In [5]:
T1 = generateTransformation()
T2 = generateTransformation()

#### Method 1: (5 points)
Linearly interpolate between the individual elements of T1 and T2

In [13]:
##############################################################################
# TODO: Implement the above
##############################################################################
import open3d as o3d

points = o3d.io.read_point_cloud("toothless.ply")
anim = animate_tf(points)
anim.setup(T1, T2)
anim.animate()




#### Question
Q: What is your opinion on this method? Would you use this in practice? Why or why not?

A: 
Method 1 — Direct Matrix Linear Interpolation

Opinion:

This is the simplest hack: just average the entries of the transformation matrices.

It does not guarantee a valid rotation (intermediate “rotation” matrices can become non-orthogonal → stretching, skewing, scaling can creep in).

So the point cloud can appear to distort unnaturally during interpolation.

Would I use it in practice?

❌ No, except maybe for quick demos or prototypes where correctness is not important.

Not used in robotics, animation, or computer graphics because it breaks the rigid-body constraint.

🔹 Method 2 — Euler Angle Linear Interpolation

Opinion:

Better than raw matrix blending, since we explicitly treat orientation as a rotation.

But Euler angles suffer from:

Gimbal lock (loss of a degree of freedom at certain angles).

Discontinuities (e.g. interpolating between 359° and 1° should be 2°, but naïve interpolation goes the long way around: 359 → 180 → 1).

Motion can look jerky or unnatural if the angles cross singularities.

Would I use it in practice?

⚠️ Rarely, and only when Euler angles are already the native representation (e.g. some CAD/graphics tools, or legacy robotics systems).

Even then, I would prefer to wrap interpolation carefully (e.g. unwrap angles to avoid discontinuities).

🔹 Method 3 — Quaternion SLERP

Opinion:

This is the gold standard for smooth, shortest-path interpolation of rotations.

Quaternions live on a unit sphere in 4D, and SLERP moves along the geodesic arc → always a valid rotation, no distortion.

Avoids gimbal lock, avoids singularities.

Translation interpolation remains linear, which is perfectly fine.

Would I use it in practice?

✅ Absolutely yes.

SLERP is the standard in:

Robotics (interpolating robot joint poses, hand–eye calibration).

Computer graphics (character animation, camera control).

AR/VR and 3D vision (pose blending).

Only downside: slightly more math than Euler, but modern libraries (SciPy, Eigen, Unity, Blender) already have it built in.

#### Method 2: (5 points)
Decompose R1 and R2 (from T1 and T2) into their Euler Angle representation and now use these in your linear interpolation, along with t1 and t2 (translations)

In [ ]:
##############################################################################
# TODO: Implement the above
##############################################################################

rot = rotation()

def euler_interpol_fn(t1,t2,alpha):
    r1 = t1[:3,:3]
    r2 = t2[:3,:3]
    
    rot.from_matrix(r1)
    e1 = rot.as_euler()

    rot.from_matrix(r2)
    e2 = rot.as_euler()

    e = (1.0 - alpha) * e1 + alpha * e2
    t = (1.0 - alpha) * t1[:3, 3] + alpha * t2[:3, 3]
    rot.from_euler(e)
    r = rot.as_matrix()

    T = np.eye(4)
    T[:3, :3] = r
    T[:3, 3] = t
    return T

points = o3d.io.read_point_cloud("toothless.ply")
anim = animate_tf(points,euler_interpol_fn)
anim.setup(T1, T2)
anim.animate()

#### Question
Q: What is your opinion on this method? Would you use this in practice? Why or why not?

A: 


#### Method 3: (5 points)
Use Slerp to interpolate between T1 and T2

In [24]:
##############################################################################
# TODO: Implement the above question using spherical linear interpolation (slerp)
##############################################################################

def slerp_interpolator(T1, T2, alpha):
    R1 = T1[:3, :3]
    R2 = T2[:3, :3]
    t1 = T1[:3, 3]
    t2 = T2[:3, 3]

    rot = rotation()
    q1 = rot.as_quat(R1)
    q2 = rot.as_quat(R2)

    dot = np.dot(q1, q2)
    if dot < 0.0:
        q2 = -q2
        dot = -dot
    dot = np.clip(dot, -1.0, 1.0)

    if dot > 0.9995:
        q_interp = (1.0 - alpha) * q1 + alpha * q2
        q_interp /= np.linalg.norm(q_interp)
    else:
        theta_0 = np.arccos(dot)
        sin_theta_0 = np.sin(theta_0)
        theta = theta_0 * alpha
        sin_theta = np.sin(theta)
        s0 = np.cos(theta) - dot * sin_theta / sin_theta_0
        s1 = sin_theta / sin_theta_0
        q_interp = (s0 * q1) + (s1 * q2)
        q_interp /= np.linalg.norm(q_interp)

    rot.from_quat(q_interp)
    R_interp = rot.as_matrix()

    t_interp = (1 - alpha) * t1 + alpha * t2

    T = np.eye(4)
    T[:3, :3] = R_interp
    T[:3, 3] = t_interp
    return T


points = o3d.io.read_point_cloud("toothless.ply")
anim = animate_tf(points,slerp_interpolator)
anim.setup(T1, T2)
anim.animate()

#### References: https://en.wikipedia.org/wiki/Slerp

#### Question
Q: What is your opinion on this method? Would you use this in practice? Why or why not?

A: 

## 3.2 - World and Body Frame Transforms (5 points)

In [ ]:
T1 = generateTransformation()
T2 = generateTransformation()

What is the single world frame transform that can be applied to T1, to convert it to T2?

Print the answer

In [19]:
##############################################################################
# TODO: Implement the above question
##############################################################################

def relative_transform(T1, T2):
    """
    Compute the world-frame transform ΔT such that T2 = ΔT @ T1
    """
    T1_inv = np.linalg.inv(T1)
    return T2 @ T1_inv

# Example
def generateTransformation():
    angle = np.random.uniform(0, 2*np.pi)
    axis = np.random.randn(3)
    axis /= np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    Rm = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    t = np.random.uniform(-5, 5, (3,))
    T = np.eye(4)
    T[:3, :3] = Rm
    T[:3, 3] = t
    return T

# Generate two random transforms
T1 = generateTransformation()
T2 = generateTransformation()

# Compute ΔT
Delta_T = relative_transform(T1, T2)

print("T1:\n", T1)
print("\nT2:\n", T2)
print("\nΔT (world-frame transform from T1 → T2):\n", Delta_T)

def check_transform(T):
    R = T[:3,:3]
    t = T[:3,3]

    # 1) Orthonormality & determinant
    ortho_err = np.linalg.norm(R.T @ R - np.eye(3))
    detR = np.linalg.det(R)

    # 2) Reconstruction error
    recon_ok = np.allclose(T2, T @ T1, atol=1e-8)

    # 3) Rotation angle (rad/deg) and unit axis
    angle = np.arccos(np.clip((np.trace(R) - 1)/2, -1.0, 1.0))
    if angle < 1e-12:
        axis = np.array([1.0, 0.0, 0.0])  # arbitrary when angle~0
    else:
        axis = np.array([
            R[2,1] - R[1,2],
            R[0,2] - R[2,0],
            R[1,0] - R[0,1],
        ]) / (2*np.sin(angle))

    print("=== ΔT checks ===")
    print("Orthonormality error ||RᵀR - I||:", ortho_err)
    print("det(R):", detR)
    print("Reconstruction T2 ≈ ΔT @ T1:", recon_ok)
    print("Rotation angle (deg):", np.degrees(angle))
    print("Rotation axis (unit):", axis)
    print("Translation (t):", t)
    print("Translation norm:", np.linalg.norm(t))

# Use your matrices
T1 = np.array([[ 0.99715552,  0.01686766,  0.07345987,  3.63735578],
               [-0.00963534,  0.99516768, -0.09771617, -2.53943846],
               [-0.07475313,  0.09673041,  0.99249947,  1.26425521],
               [ 0., 0., 0., 1.]])

T2 = np.array([[ 0.28478537, -0.71799543,  0.63512192, -4.32912473],
               [-0.70386838,  0.29315356,  0.64701646, -2.97207839],
               [-0.65074311, -0.63130306, -0.42188843,  0.06676249],
               [ 0., 0., 0., 1.]])

Delta_T = np.array([[ 0.31852037, -0.77933153,  0.53961758, -8.14897546],
                    [-0.64939168,  0.23529499,  0.72313672, -0.92672203],
                    [-0.69053256, -0.58075695, -0.43114517,  1.64875608],
                    [ 0., 0., 0., 1.]])

check_transform(Delta_T)


T1:
 [[ 0.21588658  0.93313293  0.28749943 -4.38361859]
 [ 0.55317924  0.12574501 -0.82351741 -0.78936157]
 [-0.80460283  0.33682507 -0.4890431  -3.49284683]
 [ 0.          0.          0.          1.        ]]

T2:
 [[-0.81494482 -0.54947585 -0.18423144  2.41148572]
 [ 0.35088178 -0.72081968  0.59774657 -4.14319398]
 [-0.46124496  0.42248702  0.78022933 -2.26299677]
 [ 0.          0.          0.          1.        ]]

ΔT (world-frame transform from T1 → T2):
 [[-0.74163609 -0.3681866   0.56072679  0.82833637]
 [-0.42501811 -0.38879367 -0.81743445 -9.1683834 ]
 [ 0.51897544 -0.84455794  0.13185744 -0.1941101 ]
 [ 0.          0.          0.          1.        ]]
=== ΔT checks ===
Orthonormality error ||RᵀR - I||: 1.592697570240657e-08
det(R): 1.0000000017972999
Reconstruction T2 ≈ ΔT @ T1: True
Rotation angle (deg): 116.01872781398274
Rotation axis (unit): [-0.72547301  0.68444287  0.07229719]
Translation (t): [-8.14897546 -0.92672203  1.64875608]
Translation norm: 8.365584939495058


What is the body frame equivalent of the same?

Print the answer

In [22]:
##############################################################################
# TODO: Implement the above question
##############################################################################
import numpy as np

def relative_transform_body(T1, T2):
    """
    Compute the relative transform ΔT_body = T1⁻¹ @ T2
    expressing T2 in T1's body frame.
    """
    Delta_T_body = np.linalg.inv(T1) @ T2
    R_b = Delta_T_body[:3,:3]
    t_b = Delta_T_body[:3,3]

    # --- Checks ---
    ortho_err = np.linalg.norm(R_b.T @ R_b - np.eye(3))
    detR = np.linalg.det(R_b)
    recon_ok = np.allclose(T2, T1 @ Delta_T_body, atol=1e-8)

    # --- Rotation angle & axis ---
    angle = np.arccos(np.clip((np.trace(R_b) - 1)/2, -1.0, 1.0))
    if abs(angle) < 1e-12:
        axis = np.array([1.,0.,0.])
    else:
        axis = np.array([R_b[2,1]-R_b[1,2], R_b[0,2]-R_b[2,0], R_b[1,0]-R_b[0,1]]) / (2*np.sin(angle))

    # Print results
    print("=== ΔT_body (T1⁻¹ @ T2) ===")
    print(Delta_T_body)
    print("\nChecks:")
    print("Orthonormality err:", ortho_err)
    print("det(R):", detR)
    print("Reconstruction T2 ≈ T1 @ ΔT_body:", recon_ok)
    print("Rotation angle (deg):", np.degrees(angle))
    print("Rotation axis:", axis)
    print("Translation:", t_b)
    print("Translation norm:", np.linalg.norm(t_b))

    return Delta_T_body

# Example usage with your T1, T2:
T1 = np.array(T1)
T2 = np.array(T2)

Delta_T_body = relative_transform_body(T1, T2)


=== ΔT_body (T1⁻¹ @ T2) ===
[[ 0.3394024  -0.67158586  0.65861859 -7.85013504]
 [-0.75861005  0.21855984  0.61379345 -0.68075911]
 [-0.55616257 -0.70795765 -0.43529204 -1.73145158]
 [ 0.          0.          0.          1.        ]]

Checks:
Orthonormality err: 1.000640033886869e-08
det(R): 0.9999999981924812
Reconstruction T2 ≈ T1 @ ΔT_body: True
Rotation angle (deg): 116.01872746664067
Rotation axis: [-0.73540869  0.67589172 -0.04841936]
Translation: [-7.85013504 -0.68075911 -1.73145158]
Translation norm: 8.067588097307508


## Bonus (5 points)
#### Successive Transforms
To interpolate between T1 and T2, can you find the single world frame transform that can be applied N succesive times to T1, to get T2.

In [ ]:
##############################################################################
# TODO: Implement the above question using spherical linear interpolation (slerp)
##############################################################################




Single step transform (apply 10 times):
 [[ 0.99032191 -0.02468475  0.13657665 -0.81489755]
 [ 0.00439462  0.98913998  0.14691084 -0.0926722 ]
 [-0.13871988 -0.14488882  0.97967547  0.16487561]
 [ 0.          0.          0.          1.        ]]

Reconstructed T2:
 [[ 0.28478538 -0.71799543  0.63512192 -1.47967402]
 [-0.70386838  0.29315356  0.64701646 -0.32499945]
 [-0.65074311 -0.63130306 -0.42188843  3.59975258]
 [ 0.          0.          0.          1.        ]]

Matches T2: False


## Even more Bonus (5 points)
Can you do the above in Body Frame?

In [ ]:
##############################################################################
# TODO: Implement the above question using spherical linear interpolation (slerp)
##############################################################################